
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
import json
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000261F6F35400>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000261F6F35E80>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

#### Lets build a recipe based system

In [3]:
from pydantic import BaseModel, Field


class Recipe(BaseModel):
    name:str=Field(description="The name of the dish")
    cuisine:str=Field(description="The cuisine style of the dish")
    prep_time:int=Field(description="Preparation time in minutes")
    ingredients:list[str]=Field(description="List of ingredients needed for the dish")

In [4]:
# not an model invocation, with_structured_output(Recipe) just a runnable
model_with_structure=model.with_structured_output(Recipe)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000261F6F35400>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000261F6F35E80>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Recipe', 'description': '', 'parameters': {'properties': {'name'

In [5]:
model.invoke("Tell me about khachapuri, the Georgian cheese-filled bread")

AIMessage(content='**Khachapuri – Georgia’s iconic cheese‑filled bread**\n\n> *Khachapuri* (ხაჭაპური) literally means “cheese bread” in Georgian. It’s more than a snack – it’s a culinary symbol of hospitality, tradition, and regional pride.\n\n---\n\n## 1. Quick Snapshot\n\n| Feature | Detail |\n|---------|--------|\n| **Origin** | Traditional to the Imereti region (western Georgia), now national |\n| **Main Ingredients** | Flour, yeast, water, salt, cheese (sulguni, tinotuli, or a blend) |\n| **Shape** | Boat‑shaped “lunch” version; round “adjarian” version |\n| **Serving** | Warm, often with a raw egg and butter; eaten with hands or a fork |\n| **Cultural Note** | Served at family gatherings, festivals, and Georgian restaurants worldwide |\n\n---\n\n## 2. Regional Variants\n\n| Variant | Shape & Style | Cheese | Extra Touches |\n|---------|---------------|--------|---------------|\n| **Imeretian (lunch‑khachapuri)** | Long, boat‑shaped dough with a cheese “cavity” | Sulguni, tinotuli

In [6]:
response=model_with_structure.invoke("Provide the recipe details for khachapuri, the Georgian cheese-filled bread")

print(json.dumps(response.model_dump(),indent=2, default=str))

{
  "name": "Khachapuri",
  "cuisine": "Georgian",
  "prep_time": 90,
  "ingredients": [
    "2 cups all-purpose flour",
    "1 packet active dry yeast (2 1/4 tsp)",
    "1/4 cup warm water",
    "1/4 cup warm milk",
    "1 tsp sugar",
    "1 tsp salt",
    "2 tbsp melted butter",
    "1 egg",
    "1 cup shredded sulguni cheese",
    "1 cup shredded mozzarella",
    "1 tbsp butter (for topping)",
    "1 egg (for egg wash)"
  ]
}


### MEssage output alongside parsed structure

In [7]:
from pydantic import BaseModel, Field


class Recipe(BaseModel):
    """A recipe with details."""
    name: str = Field(..., description="The name of the dish")
    cuisine: str = Field(..., description="The cuisine style of the dish")
    prep_time: int = Field(..., description="Preparation time in minutes")
    ingredients: list[str] = Field(..., description="List of ingredients needed for the dish")

model_with_structure = model.with_structured_output(Recipe, include_raw=True)  

response = model_with_structure.invoke("Provide the recipe details for khachapuri, the Georgian cheese-filled bread")


In [8]:
response
# print(json.dumps(
#     {
#         "raw": response["raw"].model_dump(),
#         "parsed": response["parsed"].model_dump(),
#         "parsing_error": response["parsing_error"],
#     },
#     indent=2,
#     default=str,
#     ensure_ascii=False,
# ))

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants recipe details for khachapuri. The tool 'functions' has a recipe type that requires cuisine, ingredients, name, prep_time. We should call that function.", 'tool_calls': [{'id': 'fc_db1466d4-2ee2-4ce5-a283-71418112c0a4', 'function': {'arguments': '{"cuisine":"Georgian","ingredients":["Flour","Water","Yeast","Sugar","Salt","Olive oil","Cheese (a mix of Sulguni and/or Imeretian cheese)","Butter"],"name":"Khachapuri","prep_time":90}', 'name': 'Recipe'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 173, 'total_tokens': 288, 'completion_time': 0.119113822, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_time': 0.103551992, 'prompt_tokens_details': None, 'queue_time': 0.304207614, 'total_time': 0.222665814}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls

### Nested Structure

In [ ]:
# This may fail if the model did not generated the output according to our pydantic model
from pydantic import BaseModel, Field


class Ingredient(BaseModel):
    item: str
    quantity: str

class RecipeDetails(BaseModel):
    name: str
    cuisine: str
    ingredients: list[Ingredient]
    steps: list[str]
    prep_time: float | None = Field(None, description="Prep time in minutes")

model_with_structure = model.with_structured_output(RecipeDetails)

response = model_with_structure.invoke("Provide the full recipe for khachapuri, the Georgian cheese-filled bread")
response

RecipeDetails(name='Khachapuri', cuisine='Georgian', ingredients=[Ingredient(item='All-purpose flour', quantity='500 g'), Ingredient(item='Warm water', quantity='300 ml'), Ingredient(item='Active dry yeast', quantity='7 g (1 packet)'), Ingredient(item='Sugar', quantity='1 tsp'), Ingredient(item='Salt', quantity='1 tsp'), Ingredient(item='Melted butter', quantity='2 tbsp'), Ingredient(item='Warm milk', quantity='200 ml'), Ingredient(item='Sulguni cheese (or a mix of feta and mozzarella)', quantity='300 g'), Ingredient(item='Egg (for the filling)', quantity='1'), Ingredient(item='Butter (for topping)', quantity='50 g'), Ingredient(item='Egg (for the top)', quantity='1')], steps=['1. In a small bowl, combine the warm water, sugar, and yeast. Let it sit for 5–10 minutes until frothy.', '2. In a large mixing bowl, whisk together the flour and salt. Add the yeast mixture, melted butter, and warm milk to the dry ingredients. Mix until a shaggy dough forms.', '3. Turn the dough onto a floured 

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [10]:
from typing import Annotated

from typing_extensions import TypedDict


class RecipeDict(TypedDict):
    """A recipe with details."""
    name: Annotated[str, ..., "The name of the dish"]
    cuisine: Annotated[str, ..., "The cuisine style of the dish"]
    prep_time: Annotated[int, ..., "Preparation time in minutes"]
    ingredients: Annotated[list[str], ..., "List of ingredients needed for the dish"]


model_withtypedict=model.with_structured_output(RecipeDict)
response=model_withtypedict.invoke("Please provide the recipe details for pizza")
response

{'cuisine': 'Italian',
 'ingredients': ['Pizza dough',
  'Tomato sauce',
  'Mozzarella cheese',
  'Olive oil',
  'Salt',
  'Pepper',
  'Fresh basil'],
 'name': 'Pizza',
 'prep_time': 60}

In [11]:
bad = RecipeDict(
    name=123,                 # should be str
    cuisine=None,             # should be str
    prep_time="not a number", # should be int
    ingredients="cheese",     # should be list[str]
)
print(bad)   # no error

{'name': 123, 'cuisine': None, 'prep_time': 'not a number', 'ingredients': 'cheese'}


In [12]:
class Ingredient(TypedDict):  # noqa: F811
    item: str
    quantity: str

class RecipeDetails(TypedDict):
    name: str
    cuisine: str
    ingredients: list[Ingredient]
    steps: list[str]
    prep_time: float | None = Field(None, description="Prep time in minutes")

model_with_structure = model.with_structured_output(RecipeDetails)

response = model_with_structure.invoke("Provide the full recipe for pancakes")
response

{'cuisine': 'American',
 'ingredients': [{'item': 'All-purpose flour', 'quantity': '1 1/2 cups'},
  {'item': 'Baking powder', 'quantity': '3 1/2 teaspoons'},
  {'item': 'Salt', 'quantity': '1 teaspoon'},
  {'item': 'White sugar', 'quantity': '1 tablespoon'},
  {'item': 'Milk', 'quantity': '1 1/4 cups'},
  {'item': 'Egg', 'quantity': '1'},
  {'item': 'Butter, melted', 'quantity': '3 tablespoons'},
  {'item': 'Vanilla extract', 'quantity': '1 teaspoon'}],
 'name': 'Classic Pancakes',
 'prep_time': 20,
 'steps': ['In a large bowl, sift together the flour, baking powder, salt, and sugar.',
  'Make a well in the center and pour in the milk, egg, melted butter, and vanilla extract.',
  'Whisk until just combined; the batter should be slightly lumpy.',
  'Heat a lightly oiled griddle or frying pan over medium-high heat.',
  'Pour or scoop the batter onto the griddle, using approximately 1/4 cup for each pancake.',
  'Cook until bubbles form on the surface and the edges look set, about 2-3 min

In [13]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2025-08-05',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [14]:
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")

In [15]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029', additional_kwargs={}, response_metadata={}, id='82608968-9481-42d0-a56c-a33b12bf4e16'),
  AIMessage(content='{"name":"Elena Marquez","email":"elena.marquez@brightleaf-consulting.com","phone":"(312) 774-5029"}', additional_kwargs={'reasoning_content': 'We need to output only the final JSON object in the ContactInfo format. Must be compact JSON. Ensure all required fields present: name, email, phone. Provide correct values.\n\nInput: "Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029"\n\nName: "Elena Marquez"\nEmail: "elena.marquez@brightleaf-consulting.com"\nPhone: "(312) 774-5029"\n\nReturn compact JSON: {"name":"Elena Marquez","email":"elena.marquez@brightleaf-consulting.com","phone":"(312) 774-5029"}'}, response_metadata={'token_usage': {'completion_tokens': 183, 'prompt_tokens': 247, 'total_tokens': 430, 'completion_time': 0.202

In [16]:
result["structured_response"]

ContactInfo(name='Elena Marquez', email='elena.marquez@brightleaf-consulting.com', phone='(312) 774-5029')

In [17]:
## Dataclass

from dataclasses import dataclass

from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029"}]
})

result["structured_response"]

ContactInfo(name='Elena Marquez', email='elena.marquez@brightleaf-consulting.com', phone='(312) 774-5029')

### dataclass and pydantic in same example

In [18]:
from dataclasses import dataclass

from pydantic import BaseModel, ValidationError


@dataclass
class ContactDC:
    name: str
    email: str
    phone: str

class ContactPD(BaseModel):
    name: str
    email: str
    phone: str

print(ContactDC(name=123, email=None, phone=["x"]))   # works, no error

try:
    ContactPD(name=123, email=None, phone=["x"])
except ValidationError as e:
    print(e)                                           # raises

ContactDC(name=123, email=None, phone=['x'])
3 validation errors for ContactPD
name
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
email
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
phone
  Input should be a valid string [type=string_type, input_value=['x'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
